# Caught & Shared — Embedding Blending Approach
**Controllable Mentor Influence + User Preference Persistence**

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data

In [ ]:
num_users = 25
num_items = 80

interactions = []
for u in range(num_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

initial_mentor_assignments = [
    (0, 5, 0.70),
    (0, 7, 0.40),
    (1, 12, 0.85),
    (2, 5, 0.65),
    (3, 8, 0.90),
    (4, 15, 0.55),
]

#### 2. Influence Presets

In [ ]:
INFLUENCE_PRESETS = {
    "Light":    {"alpha": 0.20, "label": "Light Touch",     "desc": "Slight nudge toward mentor's taste"},
    "Balanced": {"alpha": 0.45, "label": "Friendly Half",   "desc": "Almost equal mix of both tastes"},
    "Deep":     {"alpha": 0.70, "label": "Big Piece",       "desc": "Strong immersion into mentor's taste"}
}

def get_preset_alpha(preset_name: str) -> float:
    return INFLUENCE_PRESETS[preset_name]["alpha"]

#### 3. User Preferences Storage (with JSON persistence)

In [ ]:
USER_PREFERENCES_FILE = Path("user_mentor_preferences.json")
user_preferences = {}   # {user_id: {mentor_id: {"preset": , "alpha": , "added_at": }}}


def load_preferences():
    global user_preferences
    if USER_PREFERENCES_FILE.exists():
        with open(USER_PREFERENCES_FILE, 'r', encoding='utf-8') as f:
            user_preferences = json.load(f)
        print(f"Loaded preferences for {len(user_preferences)} users")
    else:
        print("No saved preferences found. Starting fresh.")


def save_preferences():
    with open(USER_PREFERENCES_FILE, 'w', encoding='utf-8') as f:
        json.dump(user_preferences, f, indent=2, ensure_ascii=False)


def save_mentor_choice(user_id: int, mentor_id: int, preset: str = None, alpha: float = None):
    if user_id not in user_preferences:
        user_preferences[user_id] = {}
    
    if preset:
        alpha = get_preset_alpha(preset)
    
    user_preferences[user_id][str(mentor_id)] = {
        "preset": preset,
        "alpha": round(float(alpha), 2),
        "added_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    save_preferences()


def remove_mentor(user_id: int, mentor_id: int):
    if user_id in user_preferences and str(mentor_id) in user_preferences[user_id]:
        del user_preferences[user_id][str(mentor_id)]
        if not user_preferences[user_id]:
            del user_preferences[user_id]
        save_preferences()
        print(f"Removed Mentor {mentor_id} from User {user_id}")
    else:
        print(f"No such mentor for User {user_id}")


def get_user_mentor_config(user_id: int):
    """Returns list of (mentor_id, alpha)"""
    if user_id not in user_preferences:
        return []
    return [(int(m_id), info["alpha"]) 
            for m_id, info in user_preferences[user_id].items()]


load_preferences()

#### 4. Recommendation Function

In [ ]:
@torch.no_grad()
def get_recommendations_blending(user_id: int, mentor_config=None, k=8):
    if mentor_config is None:
        mentor_config = get_user_mentor_config(user_id)
    
    user_emb = user_embs[user_id].clone()
    
    if mentor_config:
        final_emb = user_emb.clone()
        for mentor_id, alpha in mentor_config:
            mentor_emb = user_embs[mentor_id]
            final_emb += alpha * (mentor_emb - user_emb)
        final_emb = F.normalize(final_emb, p=2, dim=0)
    else:
        final_emb = user_emb
    
    scores = torch.matmul(final_emb.unsqueeze(0), item_embs.T).squeeze(0)
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'mentors': mentor_config,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }

In [ ]:
save_mentor_choice(user_id=0, mentor_id=5, preset="Deep")
save_mentor_choice(user_id=0, mentor_id=7, preset="Balanced")
save_mentor_choice(user_id=1, mentor_id=12, preset="Light")

print("\nCurrent preferences:")
for uid in [0, 1]:
    config = get_user_mentor_config(uid)
    print(f"User {uid}: {config}")

rec = get_recommendations_blending(user_id=0)
print(f"\nRecommendations for User 0: {rec['recommended_items']}")

remove_mentor(user_id=0, mentor_id=7)